# XGBoost

Gradient boosting baseline for LD50 regression.

In [9]:
from pathlib import Path
import sys
import warnings

import numpy as np
import xgboost as xgb

PROJECT_ROOT = Path('../..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils.modeling import artifact_paths, load_tabular_arrays, save_ml_run


In [10]:
MODEL_NAME = 'xgboost'
X_train, X_val, X_test, y_train, y_val, y_test, feature_names = load_tabular_arrays()
X_train = X_train.astype(np.float32)
X_val = X_val.astype(np.float32)
X_test = X_test.astype(np.float32)
paths = artifact_paths(Path.cwd(), MODEL_NAME, '.pkl')


In [11]:
model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    device='cuda',
    random_state=42,
)

try:
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
except Exception as exc:
    warnings.warn(f'XGBoost CUDA failed; retrying on CPU. Details: {exc}')
    model.set_params(device='cpu')
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

predictions = model.predict(X_test)


In [12]:
save_ml_run('XGBoost', model, paths, X_test, y_test, predictions, feature_names)


[XGBoost] RMSE: 0.5574 | MAE: 0.4131 | R2: 0.6522


{'rmse': 0.5574071411544532,
 'mae': 0.4130749787877521,
 'r2': 0.6522275882393463}